In [2]:
import cv2
import numpy as np
import tensorflow as tf
import config

# Load model & labels
interpreter = tf.lite.Interpreter(model_path=str(config.OUTPUT_TFLITE))
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

with open(config.OUTPUT_LABELS, "r") as f:
    labels = [line.strip() for line in f if line.strip()]

# Open PC Webcam (CAP_DSHOW for smooth Windows capture)
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

print(" Live Webcam Started!")
print(" Hold up animal photos to your webcam. Press 'q' on the camera window to stop.")

while True:
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame.")
        break

    # Preprocess frame for MobileNetV3
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (config.INPUT_WIDTH, config.INPUT_HEIGHT))
    input_data = np.expand_dims(resized, axis=0).astype(np.float32)

    # Inference
    interpreter.set_tensor(input_details[0]["index"], input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]["index"])[0]

    # Get Top Prediction
    top_idx = int(np.argmax(output))
    pred_label = labels[top_idx]
    confidence = float(output[top_idx])

    # Overlay HUD on camera feed
    color = (0, 255, 0) if confidence > 0.60 else (0, 165, 255) 
    cv2.rectangle(frame, (10, 10), (450, 80), (0, 0, 0), -1)
    cv2.putText(
        frame,
        f"{pred_label}: {confidence * 100:.1f}%",
        (25, 55),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.1,
        color,
        2,
    )

    cv2.imshow("Edge AI Animal Detection (Press 'q' to Quit)", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Webcam closed.")


 Live Webcam Started!
 Hold up animal photos to your webcam. Press 'q' on the camera window to stop.
Webcam closed.
